## __Aprendizaje no supervisado__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

__Asunto__: SVM

***

In [ ]:
## Librerias
from itertools import product
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from pandas import DataFrame
from numpy import array, ones, concatenate, linspace
from numpy.random import seed, randn, uniform

from sklearn.model_selection import train_test_split
from sklearn.svm import OneClassSVM

__Dataset:__

In [ ]:
seed(0)

## Numero de muestras, y de outliers
n_samples, n_outliers = 360, 100

## matriz de covarianza
covariance = array([[0.5, -0.1], [0.7, 0.4]])

## Generación de grupo de puntos
cluster_1 = 0.4 * randn(n_samples, 2) @ covariance + array([2, 2])  # general
cluster_2 = 0.3 * randn(n_samples, 2) + array([-2, -2])  # spherical
outliers = uniform(low=-4, high=4, size=(n_outliers, 2))

## Armado de dataset
X = concatenate([cluster_1, cluster_2, outliers])
y = concatenate(
    [ones((2 * n_samples), dtype=int), -ones((n_outliers), dtype=int)]
)

print('(shape) X: {} - y: {}'.format(X.shape, y.shape))

In [ ]:
## Partición de datos
X_train_Val, X_test, y_train_val, y_test = train_test_split(X, y, 
                                                            test_size=0.121, 
                                                            stratify=y, 
                                                            random_state=9001)

X_train, X_val, y_train, y_val = train_test_split(X_train_Val, y_train_val, 
                                                  test_size=0.138, 
                                                  stratify=y_train_val, 
                                                  random_state=9001)

print('(shape - Train) X: {} - y: {}'.format(X_train.shape, y_train.shape))
print('(shape - Validate) X: {} - y: {}'.format(X_val.shape, y_val.shape))
print('(shape - Test) X: {} - y: {}'.format(X_test.shape, y_test.shape))

__Visualización de los datos considerados__

In [ ]:
plt.figure(figsize=(7, 7))
sns.scatterplot(x=X_train[:, 0], 
                y=X_train[:,1], 
                )
plt.title("Nube de puntos")
plt.xlabel('x'), plt.ylabel('y')
plt.tight_layout()
plt.show()

## Clase OneClassSVM

```{python}
    OneClassSVM(kernel='rbf', 
                degree=3, 
                gamma='scale', 
                tol=0.001, 
                nu=0.5, 
                verbose=False, 
                max_iter=-1)
```

| Parámetros | Descripción |
|------------|-------------|
| kernel | especificar el kernel a usar, por ejemplo: 'linear', 'poly', 'rbf', 'sigmoid' (por defecto, 'rbf'). |
| degree | grado del kernel polinómica (por defecto, 3). |
| gamma | es un parámetro de escala del kernel: 'auto': 1 / (n_features * X.var()), 'scale':  1 / n_features (por defecto, 'scale'). |
| tol | margen de error en solución entre iteraciones (por defecto, 0.001) | 
| nu | es el porcentaje mal clasificada del conjunto de entrenamiento, además el porcentaje mínimo de los vectores de soportes. |
| verbose | 0 no mostrar salida de iteraciones, 1 mostrar detalles (por defecto, 0). | 
| max_iter | número máximo de iteraciones. -1 sin límite (por defecto, -1). |

<br>

| Atributos | Descripción |
|------------|-------------|
| fit_status_ | 0 si el modelo alcanzó la convergencia, 1 si no. |
| n_iter_ | número de iteraciones ejecutada. |
| n_support_ | número de vectores de soporte. | 
| support_ | retorna los índices de los vectores de soporte. |
| support_vectors_ | retorna los vectores de soporte. |

<br>

|Funciones | Descripción |
|----------|-------------|
| fit(X) | Entrena el modelo con los parametros asignados.|
| fit_predict(X) | Entrena el modelo e identifica si una observación es outlier (-1) o no (1). |
| predict(X) | Identifica si una observación es outlier (-1) o no (1). |



In [ ]:
## Instancia del modelo
model = OneClassSVM(kernel='rbf', 
                    degree=3, 
                    gamma='scale', 
                    tol=0.001, 
                    nu=0.5, 
                    verbose=False, 
                    max_iter=-1)

## Ajuste del modelo
model.fit(X_train)

## Etiquetado de las observación si es o no outliers
etiquetado = model.predict(X_train)

## Mostrar cantidad de outliers
print('Cantidad de outliers detectado: {}'.format((etiquetado == -1).sum()))

## Mostrar las etiquetas
etiquetado[:40]


#### Graficación de los puntos con etiquetas

In [ ]:
plt.figure(figsize=(8, 8))
sns.scatterplot(x=X_train[:, 0], 
                y=X_train[:,1], 
                hue=etiquetado,
                palette=sns.color_palette()[:2])
plt.legend(labels=["inliers", "outliers"], title="True class")
plt.title("Gaussian inliers with uniformly distributed outliers")
plt.tight_layout()
plt.show()

#### Si disponemos de información de los outliers provenientes de expertos. 

In [ ]:
plt.figure(figsize=(14, 7))
plt.subplot(1, 2, 1)
sns.scatterplot(x=X_train[:, 0], 
                y=X_train[:,1], 
                hue=etiquetado,
                palette=sns.color_palette()[:2])
plt.legend(labels=["inliers", "outliers"], title="True class")
plt.title("OC-SVM")

plt.subplot(1, 2, 2)
sns.scatterplot(x=X_train[:, 0], 
                y=X_train[:,1], 
                hue=y_train,
                palette=sns.color_palette()[:2]
                )
plt.legend(labels=["inliers", "outliers"], title="True class")
plt.title("Reales")
plt.xlabel('x'), plt.ylabel('y')
plt.tight_layout()
plt.show()


In [ ]:
## Calculo de rendimiento del modelo si tenemos información de los outliers reales.
NroOutlier_predicha = (etiquetado == -1).sum()
NroOutlier_verdaderos = (y_train == -1).sum()
NroOutlier_acetados = ((y_train == -1) & (etiquetado == -1)).sum()
rateOutlier_acertados = ((y_train == -1) & (etiquetado == -1)).sum() / max(NroOutlier_predicha, NroOutlier_verdaderos)

print('Cantidad de outliers detectados: {}'.format(NroOutlier_predicha))
print('Cantidad de outliers verdaderos: {}'.format(NroOutlier_verdaderos))
print('Cantidad de outliers verdaderos-detectados: {}'.format(NroOutlier_acetados))
print('Porcentaje de outliers verdaderos-detectados: {:.2f}%'.format(100*rateOutlier_acertados))

## Búsqueda de la mejor configuración

In [ ]:
## lista de hiperparametros
lista_nu = linspace(0.01, 0.5, 40)
lista_degree = range(2, 11)

## Armado de grilla
grilla = list(product(['linear', 'rbf', 'sigmoid'], lista_nu, [0])) \
         +list(product(['poly'], lista_nu, lista_degree))

## Almacenador de resultados
output = {'kernel': [], 
          'nu': [],
          'degree': [],
          'rate_train_aciertos': [],
          'rate_val_aciertos': []}

for kernel, nu, degree in tqdm(grilla):
    
    ## Instancia del modelo
    model = OneClassSVM(kernel=kernel, 
                        degree=degree, 
                        gamma='scale', 
                        tol=0.001, 
                        nu=nu, 
                        verbose=False, 
                        max_iter=-1)

    ## Ajuste del modelo y Etiquetado de las observación si es o no outliers
    model.fit(X_train)
    etiquetado_train = model.predict(X_train)
    etiquetado_val = model.predict(X_val)

    ## Porcentaje de puntos anómalos acertados con respecto a los verdaderos
    acertados_train = ((y_train == -1) & (etiquetado_train == -1)).sum() / max((etiquetado_train==-1).sum(),(y_train==-1).sum())
    acertados_val = ((y_val == -1) & (etiquetado_val == -1)).sum() / max((etiquetado_val==-1).sum(),(y_val==-1).sum())

    ## Almcenar los resultados
    output['kernel'].append(kernel)
    output['nu'].append(nu)
    output['degree'].append(degree)
    output['rate_train_aciertos'].append(acertados_train)
    output['rate_val_aciertos'].append(acertados_val)

output = DataFrame(output).sort_values(by=['rate_val_aciertos', 'nu'], ascending=False)
output.head(10)


In [ ]:
## Instancia del modelo
model = OneClassSVM(kernel='rbf', 
                    degree=0, 
                    gamma='scale', 
                    tol=0.001, 
                    nu=0.160769, 
                    verbose=False, 
                    max_iter=-1)

## Ajuste del modelo y Etiquetado de las observación si es o no outliers
model.fit(X_train_Val)
etiquetado = model.predict(X_train_Val)

## Mostrar resumen comparativo
NroOutlier_predicha = (etiquetado == -1).sum()
NroOutlier_verdaderos = (y_train_val == -1).sum()
NroOutlier_acetados = ((y_train_val == -1) & (etiquetado == -1)).sum()
rateOutlier_acertados = ((y_train_val == -1) & (etiquetado == -1)).sum() / max(NroOutlier_predicha, NroOutlier_verdaderos)

print('Cantidad de outliers detectados: {}'.format(NroOutlier_predicha))
print('Cantidad de outliers verdaderos: {}'.format(NroOutlier_verdaderos))
print('Cantidad de outliers verdaderos-detectados: {}'.format(NroOutlier_acetados))
print('Porcentaje de outliers verdaderos-detectados: {:.2f}%'.format(100*rateOutlier_acertados))

In [ ]:
## Detección de outliers en el conjunto de test
etiquetado_test = model.predict(X_test)
NroOutlier_predicha = (etiquetado_test == -1).sum()
NroOutlier_verdaderos = (y_test == -1).sum()
NroOutlier_acetados = ((y_test == -1) & (etiquetado_test == -1)).sum()
rateOutlier_acertados = ((y_test == -1) & (etiquetado_test == -1)).sum() / max(NroOutlier_predicha, NroOutlier_verdaderos)

## Mostrar resumen comparativo
print('Cantidad de outliers detectados: {}'.format(NroOutlier_predicha))
print('Cantidad de outliers verdaderos: {}'.format(NroOutlier_verdaderos))
print('Cantidad de outliers verdaderos-detectados: {}'.format(NroOutlier_acetados))
print('Porcentaje de outliers verdaderos-detectados: {:.2f}%'.format(100*rateOutlier_acertados))